# Multi-token prediction

*DeepSeek-V3 technical report, section 2.2 and section 5.4.3. Notebook 4 of 8.*

**TL;DR.** A language model is normally trained to predict only the next token at each position. Multi-token prediction (MTP) adds a second, small module that, at each position, also predicts the token *after* the next one. Two things follow. During training every position supplies more learning signal, and the model is nudged to prepare its representation for what comes later. During inference the extra module can be thrown away, or reused as a cheap **draft** of the next-but-one token, which a speculative-decoding loop verifies for free. This notebook builds the module from the paper's equations, trains with and without it, and implements the speculative decoding loop, checking that it produces **exactly** the same text as ordinary decoding.

| step | what we do |
|---|---|
| 1 | the idea, and how the token positions line up |
| 2 | the MTP module and the combined loss, built and verified |
| 3 | what is shared and what is new |
| 4 | training with and without MTP |
| 5 | using the module at inference: speculative decoding, verified lossless |

In [1]:
import sys, math, time
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))             # the notebook lives next to dsv3kit.py
import dsv3kit as K                             # tested building blocks; this notebook re-shows the code it uses

torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
C = dict(blue="#1f78b4", orange="#d95f02", green="#1b9e77", purple="#7570b3", red="#c0392b", grey="#777777")

PASS, FAIL = [], []
def check(name, ok, detail=""):
    (PASS if ok else FAIL).append(name)
    print(("PASS" if ok else "FAIL"), "-", name, detail)

print("torch", torch.__version__)

torch 2.14.1


## 1. The idea, and the bookkeeping

Ordinary training: at position $i$ the model reads tokens $t_1 \dots t_i$ and is scored on predicting $t_{i+1}$. That is one target per position.

MTP with depth $D=1$ (what V3 uses) adds a second target per position: also predict $t_{i+2}$. The paper gives two motivations: the extra target **densifies the training signal**, and it may push the model to **plan ahead**, because a representation that is good for $t_{i+2}$ must already contain information about where the sentence is going.

The way V3 does it differs from the earlier multi-token work of Gloeckle et al., which uses several independent output heads that all read the same hidden state. V3 predicts the extra tokens **sequentially and keeps the causal chain**: the module for the token after next gets the main model's hidden state *and the embedding of the token in between*, so its guess is conditioned on the actual next token rather than guessing blindly around it.

![Paper Figure 3: the main model on the left predicts the next token; MTP module 1 predicts the one after that, from the main model's hidden state plus the embedding of the next token. The embedding layer and output head are shared.](../../assets/deepseek_v3/fig3_mtp.png)

*Figure 3 of the paper. Input tokens at the bottom, target tokens at the top. Each module has a shared embedding layer, a shared output head, a linear projection and one transformer block.*

Getting the shifts right is the part people trip over, so print them. Take a sequence of $T+1 = 8$ tokens, $s_0 \dots s_7$. The main model reads $s_0 \dots s_6$ and predicts $s_1 \dots s_7$. At depth $k$, position $i$ combines the previous depth's state at $i$ with the embedding of $s_{i+k}$, and predicts $s_{i+k+1}$. Each deeper depth has one position fewer, because there are no later tokens to predict from the end.

In [2]:
T_demo = 7                                              # model input length; the sequence has T+1 = 8 tokens s0..s7
print("main model   : position i reads s_0..s_i             -> predicts s_{i+1}")
for k in (1, 2):
    n = T_demo - k
    rows = [(i, f"h_{i}  +  emb(s_{i + k})", f"s_{i + k + 1}") for i in range(n)]
    print(f"\nMTP depth {k}   ({n} positions)")
    for i, inp, tgt in rows:
        print(f"   position {i}:  input = {inp:18s} target = {tgt}")

main model   : position i reads s_0..s_i             -> predicts s_{i+1}

MTP depth 1   (6 positions)
   position 0:  input = h_0  +  emb(s_1)   target = s_2
   position 1:  input = h_1  +  emb(s_2)   target = s_3
   position 2:  input = h_2  +  emb(s_3)   target = s_4
   position 3:  input = h_3  +  emb(s_4)   target = s_5
   position 4:  input = h_4  +  emb(s_5)   target = s_6
   position 5:  input = h_5  +  emb(s_6)   target = s_7

MTP depth 2   (5 positions)
   position 0:  input = h_0  +  emb(s_2)   target = s_3
   position 1:  input = h_1  +  emb(s_3)   target = s_4
   position 2:  input = h_2  +  emb(s_4)   target = s_5
   position 3:  input = h_3  +  emb(s_5)   target = s_6
   position 4:  input = h_4  +  emb(s_6)   target = s_7


## 2. The module and the loss

For depth $k$ the paper's equations 21 to 23 are:

$$
\mathbf h'^{\,k}_i = M_k\big[\,\mathrm{RMSNorm}(\mathbf h^{k-1}_i)\;;\;\mathrm{RMSNorm}(\mathrm{Emb}(t_{i+k}))\,\big],\qquad
\mathbf h^k_{1:T-k} = \mathrm{TRM}_k(\mathbf h'^{\,k}_{1:T-k}),\qquad
P^k_{i+k+1} = \mathrm{OutHead}(\mathbf h^k_i).
$$

Read it as: normalise the previous state and the embedding of the *future* token separately, glue them, project the doubled width $2d$ back to $d$ with $M_k$, run one transformer block, and read off the prediction with the **same output head** the main model uses. For $k=1$, $\mathbf h^{0}$ is the main model's final hidden state.

The training loss (eqs. 24 and 25) is the main cross-entropy plus the average of the MTP cross-entropies, weighted by $\lambda$ (0.3 for most of V3's training, 0.1 at the end):

$$ \mathcal L = \mathcal L_{main} + \frac{\lambda}{D}\sum_{k=1}^{D}\mathcal L^k_{MTP}. $$

The module and the loss as implemented in the kit (the language model that hosts them is the one from notebooks 1 to 3):

In [3]:
from dsv3kit import RMSNorm, TinyLM, LMConfig, make_block, Rope, MHA, Expert, DenseFFN, Block

class MTPModule(nn.Module):
    """One multi-token-prediction module (paper eqs. 21-22): RMSNorm both inputs, concat, project 2d->d, one block."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.n_h, self.n_e = RMSNorm(cfg.d), RMSNorm(cfg.d)
        self.proj = nn.Linear(2 * cfg.d, cfg.d, bias=False)                           # M_k in R^{d x 2d}
        self.block = make_block(cfg, layer=0)                                         # TRM_k (dense FFN here)

    def forward(self, h_prev, emb_future):
        return self.block(self.proj(torch.cat([self.n_h(h_prev), self.n_e(emb_future)], -1)))

In [4]:
def mtp_forward(model, seq, h=None):
    """Run the MTP chain.  seq (B, T+1) ids; h = trunk output on seq[:, :-1].

    Depth k works on positions i = 0 .. T-k-1: it combines the previous depth's hidden state at i with the
    embedding of token i+k (the token that is *already known* when we predict token i+k+1), eqs. (21)-(23).
    Returns [(logits_k, targets_k), ...].
    """
    T = seq.shape[1] - 1
    h_prev = model.trunk(seq[:, :-1]) if h is None else h
    out = []
    for k, mod in enumerate(model.mtp, start=1):
        n = T - k                                                                    # each depth is one position shorter
        h_k = mod(h_prev[:, :n], model.emb(seq[:, k:k + n]))                          # shared embedding
        out.append((model.head(h_k), seq[:, k + 1:k + 1 + n]))                        # shared output head
        h_prev = h_k
    return out


def lm_loss(model, seq, seq_balance_alpha: float = 0.0, batch_balance_alpha: float = 0.0):
    """Total loss = L_main + lambda/D * sum_k L^k_MTP (+ optional sequence-wise balance loss)."""
    cfg = model.cfg
    h = model.trunk(seq[:, :-1])
    main = F.cross_entropy(model.head(h).reshape(-1, cfg.vocab), seq[:, 1:].reshape(-1))
    parts, total = {"main": main.detach()}, main
    mtp = [F.cross_entropy(lg.reshape(-1, cfg.vocab), tg.reshape(-1)) for lg, tg in mtp_forward(model, seq, h)]
    for k, l in enumerate(mtp, start=1):
        parts[f"mtp{k}"] = l.detach()
    if mtp:
        total = total + cfg.mtp_lambda * torch.stack(mtp).mean()                      # eq. (25)
    if seq_balance_alpha > 0:
        bal = sum(seq_balance_loss(m, seq.shape[0], seq.shape[1] - 1, seq_balance_alpha) for m in model.moes())
        parts["seq_bal"] = bal.detach()
        total = total + bal
    if batch_balance_alpha > 0:                                                       # same loss, but over the whole batch
        bal = sum(seq_balance_loss(m, 1, seq.shape[0] * (seq.shape[1] - 1), batch_balance_alpha) for m in model.moes())
        parts["batch_bal"] = bal.detach()
        total = total + bal
    return total, parts

from dsv3kit import seq_balance_loss

Four checks. First, the shifts must be the ones in the table above. Second, the module must be what equations 21 and 22 say, computed independently. Third, nothing may leak from the future: changing tokens *after* the information a position is allowed to see must not change its prediction. Fourth, the total loss must be exactly eq. 25.

In [5]:
import dsv3kit
torch.manual_seed(0)
cfg = K.LMConfig(vocab=30, d=32, n_layers=2, n_heads=2, d_ff=48, mtp_depth=2)
model = K.TinyLM(cfg).eval()
seq = torch.randint(0, 30, (3, 9))                          # T + 1 = 9 tokens, T = 8

with torch.no_grad():
    outs = mtp_forward(model, seq)
    # 1) shapes and targets
    print("positions per depth:", [o[0].shape[1] for o in outs], " (T = 8: depth k has T-k)")
    check("depth k predicts tokens s_{k+1}..s_T", torch.equal(outs[0][1], seq[:, 2:9]) and torch.equal(outs[1][1], seq[:, 3:9]))

    # 2) independent computation of depth 1 from eqs. (21)-(23)
    h = model.trunk(seq[:, :-1])
    m1 = model.mtp[0]
    glued = torch.cat([m1.n_h(h[:, :7]), m1.n_e(model.emb(seq[:, 1:8]))], -1)             # [RMSNorm(h_i) ; RMSNorm(Emb(s_{i+1}))]
    h1 = m1.block(glued @ m1.proj.weight.T)                                                # M_1 then the transformer block
    manual = model.head(h1)
    check("module equals equations 21-23", torch.allclose(manual, outs[0][0], atol=1e-6))

    # 3) no leakage: MTP position i may see tokens up to s_{i+k}; change everything after that
    seq2 = seq.clone(); seq2[:, 6:] = torch.randint(0, 30, (3, 3))
    o1, o2 = mtp_forward(model, seq)[0][0], mtp_forward(model, seq2)[0][0]
    # depth 1, position i uses tokens s_0..s_{i+1}; positions 0..4 use only s_0..s_5 which did not change
    check("depth-1 predictions at positions 0..4 ignore tokens s_6 onward", torch.allclose(o1[:, :5], o2[:, :5], atol=1e-5))
    check("later positions do change (so the test can fail)", not torch.allclose(o1[:, 5:], o2[:, 5:], atol=1e-5))

    # 4) the total loss
    total, parts = lm_loss(model, seq)
    expected = parts["main"] + cfg.mtp_lambda / 2 * (parts["mtp1"] + parts["mtp2"])
    check("total = main + lambda/D * sum of MTP losses (eq. 25)", abs(float(total) - float(expected)) < 1e-5, f"({float(total):.4f} vs {float(expected):.4f})")

positions per depth: [7, 6]  (T = 8: depth k has T-k)


PASS - depth k predicts tokens s_{k+1}..s_T 
PASS - module equals equations 21-23 


PASS - depth-1 predictions at positions 0..4 ignore tokens s_6 onward 
PASS - later positions do change (so the test can fail) 
PASS - total = main + lambda/D * sum of MTP losses (eq. 25) (4.4173 vs 4.4173)


## 3. What is shared and what is new

The module reuses the main model's **embedding table and output head**: its only new parameters are the two RMSNorms, the projection $M_k$ (a $2d \to d$ matrix) and one transformer block. The paper notes that this sharing lets MTP gradients also train the embedding and head, and that in their pipeline the two ends of the model sit on the same device so the sharing is physical. Count what the module adds to our small model:

In [6]:
cfg1 = K.LMConfig(vocab=96, d=96, n_layers=3, n_heads=4, d_ff=192, mtp_depth=1)
m_mtp = K.TinyLM(cfg1)
m_base = K.TinyLM(K.LMConfig(vocab=96, d=96, n_layers=3, n_heads=4, d_ff=192, mtp_depth=0))
n_all = lambda m: sum(p.numel() for p in m.parameters())
n_mod = sum(p.numel() for p in m_mtp.mtp.parameters())
print(f"base model {n_all(m_base):,} parameters;   with one MTP module {n_all(m_mtp):,}   (module adds {n_mod:,} = {100 * n_mod / n_all(m_base):.0f}%)")
print("embedding / output head are shared: the module owns", [n for n, _ in m_mtp.mtp.named_parameters() if "emb" in n] or "no embedding or head parameters")
check("the module owns no embedding or head weights", not [n for n, _ in m_mtp.mtp.named_parameters() if "emb" in n or "head" in n])
check("emb weight tying: the head is the transposed embedding", m_mtp.head(torch.zeros(1, 96)).shape[-1] == 96)

base model 286,368 parameters;   with one MTP module 397,344   (module adds 110,976 = 39%)
embedding / output head are shared: the module owns no embedding or head parameters
PASS - the module owns no embedding or head weights 
PASS - emb weight tying: the head is the transposed embedding 


In V3 the module is one transformer block on top of 61, about 1.5% of the network at this ratio, and it costs nothing at inference if you discard it.

## 4. Does it help training?

The paper's ablation (its Table 4, 15.7B and 228.7B-parameter models on 1.33T and 540B tokens) reports a benefit on most benchmarks. We test the same thing, in a form we can afford: train the same small model with $D=0$ and with $D=1$ ($\lambda=0.3$), and compare the **main head's** held-out next-token loss. At inference the MTP module is discarded, so that is the only quantity that matters for the model's ordinary use. 300 steps of 48 sequences of 64 characters, two seeds.

In [7]:
data = K.CharData()
base = dict(vocab=data.vocab, d=96, n_layers=3, n_heads=4, d_ff=192)
configs = {"no MTP": K.LMConfig(**base, mtp_depth=0), "MTP depth 1": K.LMConfig(**base, mtp_depth=1, mtp_lambda=0.3)}
res = {}; models = {}
t0 = time.time()
for name, cfg in configs.items():
    rows = []
    for seed in range(2):
        model, _ = K.train_lm(cfg, data, steps=300, B=48, T=64, lr=4e-3, seed=seed)
        ev = K.evaluate(model, data, T=64, n_batches=10)
        rows.append((ev["main"], ev.get("mtp1", float("nan"))))
        models[(name, seed)] = model
    res[name] = np.array(rows)
    print(f"{name:12s} main-head val loss {res[name][:, 0].mean():.3f} +- {res[name][:, 0].std():.3f}   MTP-head val loss {res[name][:, 1].mean():.3f}")
print("time %.0f s" % (time.time() - t0))

no MTP       main-head val loss 2.449 +- 0.009   MTP-head val loss nan


MTP depth 1  main-head val loss 2.445 +- 0.044   MTP-head val loss 2.501
time 262 s


In [8]:
a, b = res["no MTP"][:, 0], res["MTP depth 1"][:, 0]
print(f"main-head loss, MTP minus no-MTP: {b.mean() - a.mean():+.3f} nats  (seed spreads {a.std():.3f}, {b.std():.3f})")
mtp_head = res["MTP depth 1"][:, 1].mean()
print(f"the MTP head's own loss ({mtp_head:.3f}) is higher than the main head's ({b.mean():.3f}): guessing two tokens ahead is harder")
check("the MTP head (two tokens ahead) has a higher loss than the main head", mtp_head > b.mean())
check("both models learn", a.mean() < math.log(data.vocab) - 1.0 and b.mean() < math.log(data.vocab) - 1.0)

main-head loss, MTP minus no-MTP: -0.004 nats  (seed spreads 0.009, 0.044)
the MTP head's own loss (2.501) is higher than the main head's (2.445): guessing two tokens ahead is harder
PASS - the MTP head (two tokens ahead) has a higher loss than the main head 
PASS - both models learn 


Compare the gap to the seed spread before reading anything into it. A character-level model that has seen 300 steps has little "planning" to do, so we do not expect to see the paper's benefit here, and we make no claim either way. What the experiment does establish is that the module trains stably alongside the main loss, that its guesses are meaningfully worse than the main head's (as they should be), and that the main head is not damaged.

## 5. Using the module at inference: speculative decoding

Decoding normally costs one full forward pass of the big model per generated token. **Speculative decoding** tries to produce two tokens per pass: guess the next-but-one token cheaply, then let the big model *verify* the guess in the same pass in which it computes its normal next token. A correct guess is accepted for free; a wrong one is discarded, and the output is unchanged.

The MTP module is exactly such a guesser: at the last position it takes the main model's state and the embedding of the token the main model has just chosen, and proposes the token after that. The loop, with the greedy version of the verification, keeps a list of committed tokens, one **pending** token $x$ that is certain, and one **draft** $d$ for the token after it:

1. Run the main model on `committed + [x, d]`.
2. The prediction $y$ at the position of $x$ is the true next token after $x$.
3. If $y = d$: **accept**. Commit $x$ and $d$; the prediction at the position of $d$ is the new pending token (a free bonus). New draft: the MTP module at $d$'s position.
4. If $y \ne d$: **reject**. Commit $x$; the new pending token is $y$. New draft: the MTP module at $x$'s position.

Each pass commits one or two tokens. The key property is that nothing about the output changes: every committed token is one the main model itself would choose. We implement it and test exactly that, on the MTP model we just trained, by comparing with plain greedy decoding.

In [9]:
@torch.no_grad()
def greedy(model, prompt, n_new):
    toks = list(prompt)
    for _ in range(n_new):
        logits = model(torch.tensor([toks]))
        toks.append(int(logits[0, -1].argmax()))
    return toks, n_new                                           # n_new forward passes

@torch.no_grad()
def mtp_draft(model, h, toks, p):
    """Guess for token p+2 from MTP position p: combine the main state h_i with Emb(token i+1) for all i <= p (the block is causal)."""
    mod = model.mtp[0]
    emb = model.emb(torch.tensor([toks[1:p + 2]]))
    h1 = mod(h[:, :p + 1], emb)
    return int(model.head(h1)[0, -1].argmax())

@torch.no_grad()
def speculative(model, prompt, n_new):
    committed = list(prompt)
    h = model.trunk(torch.tensor([committed]))                   # pass 1: just the prompt
    x = int(model.head(h)[0, -1].argmax())                       # pending token: certain
    d = mtp_draft(model, h, committed + [x], len(committed) - 1)  # draft for the token after x
    passes, accepted, tried = 1, 0, 0
    while len(committed) < len(prompt) + n_new:
        L = len(committed)
        h = model.trunk(torch.tensor([committed + [x, d]]))      # one pass over committed + pending + draft
        passes += 1
        logits = model.head(h)[0]
        y = int(logits[L].argmax())                              # true token after x
        tried += 1
        if y == d:                                               # accept: commit x and d, plus a free bonus token
            accepted += 1
            committed += [x, d]
            x = int(logits[L + 1].argmax())
            d = mtp_draft(model, h, committed + [x], L + 1)
        else:                                                    # reject: commit x, continue from y
            committed += [x]
            x = y
            d = mtp_draft(model, h, committed + [x], L)
    return committed[:len(prompt) + n_new], passes, accepted, tried

model = models[("MTP depth 1", 0)].eval()
prompts = [data.val[i:i + 32].tolist() for i in (0, 5000, 20000, 30000)]
N_NEW = 80
tot_pass = tot_tok = acc = tried = 0
identical = []
for pr in prompts:
    g, g_pass = greedy(model, pr, N_NEW)
    s, s_pass, a_, t_ = speculative(model, pr, N_NEW)
    identical.append(g == s)
    tot_pass += s_pass; tot_tok += N_NEW; acc += a_; tried += t_
print("speculative output identical to greedy on every prompt:", all(identical), identical)
print(f"draft acceptance rate      : {100 * acc / tried:.0f}%   ({acc} of {tried} drafts)")
print(f"tokens per forward pass    : {tot_tok / tot_pass:.2f}   (greedy decoding: 1.00)")
print("sample:", repr(data.decode(s[-60:])))

speculative output identical to greedy on every prompt: True [True, True, True, True]
draft acceptance rate      : 73%   (135 of 185 drafts)
tokens per forward pass    : 1.69   (greedy decoding: 1.00)
sample: 'amples the sample the sample the sample the point theand ano'


In [10]:
rate = acc / tried
print(f"relation: tokens per pass = 1 + acceptance rate  ->  1 + {rate:.2f} = {1 + rate:.2f}; measured {tot_tok / tot_pass:.2f}")
check("speculative decoding reproduces greedy decoding exactly", all(identical))
check("tokens per pass is between 1 and 2", 1.0 <= tot_tok / tot_pass <= 2.0)
check("tokens per pass is about 1 + acceptance rate (the first pass and the end effects aside)", abs(tot_tok / tot_pass - (1 + rate)) < 0.15, f"({tot_tok / tot_pass:.2f} vs {1 + rate:.2f})")

relation: tokens per pass = 1 + acceptance rate  ->  1 + 0.73 = 1.73; measured 1.69
PASS - speculative decoding reproduces greedy decoding exactly 
PASS - tokens per pass is between 1 and 2 
PASS - tokens per pass is about 1 + acceptance rate (the first pass and the end effects aside) (1.69 vs 1.73)


The identity check is the important one: speculative decoding with the MTP module is a pure speed optimisation. How much it helps is set by the acceptance rate $a$, since each pass yields $1 + a$ tokens on average. The paper reports an acceptance rate of 85 to 90% for the second token across topics, which would give about $1.85$ to $1.9$ tokens per pass and, after the cost of the extra module, the 1.8x decoding speed it quotes. Our run measured 73% (printed above), which looks close but should not be compared with the paper: greedy decoding from a tiny character model quickly falls into a repeating loop (see the printed sample, 'the sample the sample ...'), and repeated text is easy to guess. The number is a property of the toy and its degenerate output, not evidence about V3.

Two simplifications to know about: we recompute the whole sequence every pass (no KV cache), which costs time but not correctness; and we count forward passes rather than seconds, since the extra module's cost is small and our code is not optimised.

## What to take away

1. **MTP adds a second target per position** from a small module that sees the main hidden state *and the embedding of the next token*. The chain stays causal.
2. **The module shares the embedding and output head** with the main model and adds one transformer block and one projection.
3. **At inference it can be discarded** (the main model is unchanged) **or reused as the draft** in speculative decoding, which we verified is lossless.
4. **Tokens per pass is $1 + a$**, where $a$ is the draft acceptance rate.

## What this notebook does not show
- **That MTP improves quality.** The paper shows a benefit on most benchmarks at 15.7B and 228.7B parameters. Our two-seed toy (300 steps, characters) can only confirm that it trains stably and leaves the main head intact; look at the printed gap against the seed spread.
- **V3's acceptance rate or speed-up.** Ours is measured on a toy model, in forward passes, without a KV cache.
- **Deeper chains.** V3 uses depth 1; the code supports more, and the shifts are tested for depth 2, but we did not train with it.

*Next:* notebook 5 builds the FP8 training scheme.

In [11]:
print("passed:", len(PASS), "| failed:", len(FAIL))
assert not FAIL, FAIL

passed: 12 | failed: 0
